# Basic Rotation Gates

Single-qubit rotation gates.

A single qubit's state can be mapped to the Bloch sphere. Rotations around the three
axes are generated by the pauli matrices:
$$
R_a(\theta) = \exp(-i \theta / 2 \sigma_a)
$$
where $\sigma_a$ is one of the Pauli $X, Y, Z$ operators.

Since global phase is often irrelevant, practitioners can use an alternative phase convention
and define single qubit gates as the real-valued power of a Pauli operator:
$$
(\sigma_a)^t
$$
which can be multiplied by $e^{-i \pi t / 2}$ to recover the $R_a(t \pi)$ matrix.

In Qualtran, we provide `ZPowGate` and `Rz` for the two respective phase conventions, as well
as analogues for the X and Y axes.

Global phase becomes a relevant, relative phase when forming controlled gates. Indeed, the
`ZPowGate` is a controlled `GlobalPhase` operation. Whereas `ZPowGate` and `Rz` are the
same up to global phase, their controlled versions `CZPowGate` and `CRz` are different operations
with different costs.

#### General References
 - [Quantum Computation and Quantum Information](https://doi.org/10.1017/CBO9780511976667).
   Nielsen and Chuang. 2010. Section 4.2
 - [Elementary gates for quantum computation](https://arxiv.org/abs/quant-ph/9503016).
   Barenco et. al. 1995.

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from cmath import exp
from collections import Counter
from io import BytesIO
from math import pi

import torch
from IPython.display import Image, Markdown, SVG, display
from matplotlib import pyplot as plt
from sympy import Symbol

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_decomposition import decomposed_resource_callees
from flagquantum.compiler._component_trace.resource_generalization import ControlledZPowerCall, ControlledZRotationCall, RotationCall
from flagquantum.compiler.registers.primitives.controlled_rotation import controlled_rz_ir, controlled_z_power_ir
from flagquantum.compiler.registers.primitives.pauli_rotation import pauli_power_ir, pauli_rotation_ir
from flagquantum.core.ir.program.register_types import quantum_bit_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic
from flagquantum.simulation.unitary import get_unitary

## `ZPowGate`
Apply a power of the Pauli Z operator to a single qubit.

Given `exponent` $t$, the unitary matrix of this gate is:

$$
Z^t =
\begin{bmatrix}
    1 & 0 \\
    0 & e^{i \pi t}
\end{bmatrix}
$$

This is an atomic bloq in Qualtran. For many architectures, you will likely need to
synthesize an arbitrary-angle rotation from a discrete gateset like Clifford+T. Please
see the references for more information.

#### Relationships
This gate differs by a global phase from the $R_Z$ gate. `ZPowGate(t)` equals
`Rz(angle=t*np.pi)` plus `GlobalPhase(t/2)`.

This gate is the controlled version of a global phase gate. `ZPowGate(t)` equals
`GlobalPhase(t).controlled()`.

`exponent=1` corresponds to `ZGate`, `exponent=0.5` to `SGate`, and `exponent=0.25` to
`TGate`.

#### Parameters
 - `exponent`: The exponent t in Z^t.
 - `eps`: The precision of the rotation. This parameter is for bookkeeping and does not affect e.g. the tensor representation of this gate. When synthesizing a rotation from a discrete gate set, you must fix a precision `eps`. 

#### Registers
 - `q`: The qubit. 

#### References
 - [Optimal ancilla-free Clifford+T approximation of z-rotations](https://arxiv.org/abs/1403.2975). Ross and Selinger. 2014.
 - [Efficient synthesis of universal Repeat-Until-Success circuits](https://arxiv.org/abs/1404.5320). Bocharov et. al. 2014. Offers a small improvement in Cliffod+T synthesis.


In [ ]:
one_qubit = Signature.build(q=1)

def show_signature(name, signature):
    display(SVG(field_signature_svg([(name, signature, {})])))

def show_graph(key):
    children = {} if isinstance(key, RotationCall) else dict(decomposed_resource_callees(key, None))
    graph = symbolic_call_graph(key, {key: children}, max_depth=1)
    display(SVG(render_call_graph_svg(export_call_graph_dot(key, graph.direct_calls))))
    display(Markdown(format_call_graph_markdown(key, graph.direct_calls, graph.leaf_counts)))

def show_score(wires, steps):
    widths = dict.fromkeys((*wires, *(port for step in steps for port in step.outputs)), 1)
    score = layout_register_score(widths, wires, steps)
    fig, _ = draw_register_score(register_score_data(score))
    with BytesIO() as png:
        fig.savefig(png, format="png", bbox_inches="tight")
        display(Image(data=png.getvalue()))
    plt.close(fig)

### Example Instances

In [ ]:
z_pow = RotationCall("zpow", 0.123, 1e-8)

#### Graphical Signature

In [ ]:
show_signature("ZPow", one_qubit)

## `CZPowGate`
The controlled `ZPowGate`

The unitary matrix of `CZPowGate(exponent=t)` is:

$$
C[Z^t] =
\begin{bmatrix}
    1 & 0 & 0 & 0 \\
    0 & 1 & 0 & 0 \\
    0 & 0 & 1 & 0 \\
    0 & 0 & 0 & e^{i \pi t} \\
\end{bmatrix}
$$

#### Relationships
This gate has the same unitary as `Controlled(ZPowGate)`. CZPowGate(exponent=1) corresponds
to a `CZ` gate.

#### Parameters
 - `exponent`: The exponent t in Z^t.
 - `eps`: The precision of the controlled rotation. This parameter is for bookkeeping and does not affect e.g. the tensor representation of this gate. When synthesizing a rotation from a discrete gate set, you must fix a precision `eps`. 

#### Registers
 - `q`: A shape=(2,) register of two qubits ordered. This is a symmetric gate. 

#### References
 - [Simulating chemistry efficiently on fault-tolerant quantum computers](https://arxiv.org/abs/1204.0567). Jones et. al. 2012. Figure 8.


In [ ]:
cz_signature = Signature((Register("q", quantum_bit_type(), shape=(2,)),))

### Example Instances

In [ ]:
cz_pow = ControlledZPowerCall(0.123)

#### Graphical Signature

In [ ]:
show_signature("CZPow", cz_signature)

### Call Graph

In [ ]:
show_graph(cz_pow)

### Decomposition

In [ ]:
controls = ("q[0]", "q[1]")
work = "work"
show_score(controls, (
    RegisterStep("And", controls, (*controls, work), "and", (1, 1)),
    RegisterStep("ZPow", (work,), (work,)),
    RegisterStep("And†", (*controls, work), controls, "and", (1, 1)),
))

## `Rz`
Apply a single-qubit Z rotation.

Given `angle` $\theta$, the unitary matrix of this gate is:

$$
R_Z(\theta) = \exp(-i \frac{\theta}{2} Z) =
\begin{bmatrix}
    e^{-i \theta/2} & 0 \\
    0 & e^{i \theta/2}
\end{bmatrix}
$$

This is an atomic bloq in Qualtran. For many architectures, you will likely need to
synthesize an arbitrary-angle rotation from a discrete gateset like Clifford+T. Please
see the references for more information.

#### Relationships
This gate differs by a global phase from the `Z^t` gate. `Rz(a)` equals
`ZPowGate(exponent=a/np.pi)` plus `GlobalPhase(-a/(2*np.pi))`.

#### Parameters
 - `angle`: The rotation angle in radians.
 - `eps`: The precision of the rotation. This parameter is for bookkeeping and does not affect e.g. the tensor representation of this gate. When synthesizing a rotation from a discrete gate set, you must fix a precision `eps`. 

#### Registers
 - `q`: One-bit register. 

#### References
 - [Optimal ancilla-free Clifford+T approximation of z-rotations](https://arxiv.org/abs/1403.2975). Ross and Selinger. 2014.
 - [Efficient synthesis of universal Repeat-Until-Success circuits](https://arxiv.org/abs/1404.5320). Bocharov et. al. 2014. Offers a small improvement in Cliffod+T synthesis.


In [ ]:
rz_signature = one_qubit

### Example Instances

In [ ]:
a = Symbol("a")
rz = RotationCall("rz", a)

#### Graphical Signature

In [ ]:
show_signature("Rz(a)", rz_signature)

## `CRz`
A controlled Rz rotation.

Given `angle` $\theta$, the unitary matrix of this gate is:

$$
C[R_Z(\theta)] =
\begin{bmatrix}
    1 & & &  \\
    & 1 & &  \\
    & & e^{-i \theta/2} &  \\
    & & &  e^{i \theta/2}
\end{bmatrix}
$$

#### Parameters
 - `angle`: The rotation angle in radians.
 - `eps`: The precision of the rotation. This parameter is for bookkeeping and does not affect e.g. the tensor representation of this gate. When synthesizing a rotation from a discrete gate set, you must fix a precision `eps`. 

#### Registers
 - `ctrl`: Whether the rotation is active.
 - `q`: The qubit on which we optionally perform the rotation. 

#### References
 - [Elementary gates for quantum computation](https://arxiv.org/abs/quant-ph/9503016). Barenco et al. 1995. Special case of Lemma 5.4.
 - [Is Controlled(Rz(theta)) more expensive than Controlled(Z^t) on the surface code?](https://quantumcomputing.stackexchange.com/a/40012). Adam Zalcman. 2024.


In [ ]:
crz_signature = Signature.build(ctrl=1, q=1)

### Example Instances

In [ ]:
theta = Symbol(r"\theta")
crz = ControlledZRotationCall(theta)

#### Graphical Signature

In [ ]:
show_signature("CRz", crz_signature)

### Call Graph

In [ ]:
show_graph(crz)

### Decomposition

In [ ]:
ports = ("ctrl", "q")
show_score(ports, (
    RegisterStep("ZPow(+)", ("q",), ("q",)),
    RegisterStep("CX", ports, ports, "cx"),
    RegisterStep("ZPow(−)", ("q",), ("q",)),
    RegisterStep("CX", ports, ports, "cx"),
))

## `XPowGate`
A gate that rotates around the X axis of the Bloch sphere.

The unitary matrix of `XPowGate(exponent=t, global_shift=s)` is:

$$
e^{i \pi t (s + 1/2)}
\begin{bmatrix}
  \cos(\pi t /2) & -i \sin(\pi t /2) \\
  -i \sin(\pi t /2) & \cos(\pi t /2)
\end{bmatrix}
$$

Note in particular that this gate has a global phase factor of
$e^{i \pi t / 2}$ vs the traditionally defined rotation matrices
about the Pauli X axis. See `Rx` for rotations without the global
phase. The global phase factor can be adjusted by using the `global_shift`
parameter when initializing.

#### Parameters
 - `exponent`: The t in gate**t. Determines how much the eigenvalues of the gate are phased by. For example, eigenvectors phased by -1 when `gate**1` is applied will gain a relative phase of e^{i pi exponent} when `gate**exponent` is applied (relative to eigenvectors unaffected by `gate**1`).
 - `global_shift`: Offsets the eigenvalues of the gate at exponent=1. In effect, this controls a global phase factor on the gate's unitary matrix. The factor for global_shift=s is:     exp(i * pi * s * t)
 - `eps`: precision for implementation of rotation. 

#### Registers
 - `q`: One-bit register. 

#### References
 - [Efficient synthesis of universal Repeat-Until-Success circuits](https://arxiv.org/abs/1404.5320). Offers a small improvement
 - [Optimal ancilla-free Clifford+T approximation of z-rotations](https://arxiv.org/pdf/1403.2975.pdf). 


In [ ]:
x_signature = one_qubit

### Example Instances

In [ ]:
x_pow = RotationCall("xpow", 0.123, 1e-8)

#### Graphical Signature

In [ ]:
show_signature("XPow", x_signature)

### Call Graph

In [ ]:
show_graph(x_pow)

## `YPowGate`
A gate that rotates around the Y axis of the Bloch sphere.

The unitary matrix of `YPowGate(exponent=t)` is:

$$
    \begin{bmatrix}
        e^{i \pi t /2} \cos(\pi t /2) & - e^{i \pi t /2} \sin(\pi t /2) \\
        e^{i \pi t /2} \sin(\pi t /2) & e^{i \pi t /2} \cos(\pi t /2)
    \end{bmatrix}
$$

Note in particular that this gate has a global phase factor of
$e^{i \pi t / 2}$ vs the traditionally defined rotation matrices
about the Pauli Y axis. See `Ry` for rotations without the global
phase. The global phase factor can be adjusted by using the `global_shift`
parameter when initializing.

#### Parameters
 - `exponent`: The t in gate**t. Determines how much the eigenvalues of the gate are phased by. For example, eigenvectors phased by -1 when `gate**1` is applied will gain a relative phase of e^{i pi exponent} when `gate**exponent` is applied (relative to eigenvectors unaffected by `gate**1`). 
 - `global_shift`: Offsets the eigenvalues of the gate at exponent=1. In effect, this controls a global phase factor on the gate's unitary matrix. The factor for global_shift=s is:     exp(i * pi * s * t)
 - `eps`: precision for implementation of rotation. 

#### Registers
 - `q`: One-bit register. 

#### References
 - [Efficient synthesis of universal Repeat-Until-Success circuits](https://arxiv.org/abs/1404.5320). Offers a small improvement
 - [Optimal ancilla-free Clifford+T approximation of z-rotations](https://arxiv.org/pdf/1403.2975.pdf). 


In [ ]:
y_signature = one_qubit

### Example Instances

In [ ]:
y_pow = RotationCall("ypow", 0.123, 1e-8)

#### Graphical Signature

In [ ]:
show_signature("YPow", y_signature)

### Call Graph

In [ ]:
show_graph(y_pow)

## Bind and append exact rotations

Compare the full complex matrices for all three Pauli powers against their radian forms with the phase restored. Bind the symbolic CRz angle to a concrete number. For CZPow, verify both measured cleanup outcomes against the exact diagonal action and inspect the zeroed work wire. Report the emitted instructions alongside the declared cost for each controlled gate.

In [ ]:
dtype = torch.complex128

def append_unitary(width, instructions):
    circuit = Circuit(width, dtype=dtype)
    return get_unitary(circuit.append(FieldOperation(width, instructions)))

for axis, key in (("x", x_pow), ("y", y_pow), ("z", z_pow)):
    exponent = key.parameter
    angle = pi * exponent
    power = append_unitary(1, pauli_power_ir(axis, 0, exponent))
    radian = append_unitary(1, pauli_rotation_ir(axis, 0, angle))
    torch.testing.assert_close(power, exp(0.5j * angle) * radian, atol=2e-12, rtol=0)

angle = 0.37
r_gates = controlled_rz_ir(0, 1, angle)
r_expected = torch.diag(torch.tensor([1, 1, exp(-0.5j * angle), exp(0.5j * angle)], dtype=dtype))
torch.testing.assert_close(append_unitary(2, r_gates), r_expected, atol=2e-12, rtol=0)

initial = torch.tensor([1, 0, 2j, 0, 3, 0, 4j, 0], dtype=dtype)
initial /= initial.norm()
expected = initial.clone()
expected[6] *= exp(1j * pi * cz_pow.exponent)
z_gates = controlled_z_power_ir(0, 1, cz_pow.exponent, 2, 0)
dynamic = DynamicCircuit(3, inputs=initial, dtype=dtype)
dynamic.append(FieldOperation(3, z_gates))
result = run_dynamic(dynamic, shots=32, seed=19, strategy="trajectory")
assert set(result.classical_bits[:, 0].tolist()) == {0, 1}
torch.testing.assert_close(result.final_states.reshape(32, 8), expected.expand(32, 8), atol=2e-12, rtol=0)
print("CRz emitted", dict(Counter(g.name for g in r_gates)))
print("CRz model", query_qec_gate_cost(ControlledZRotationCall(angle)))
print("CZPow emitted", dict(Counter(g.name for g in z_gates)))
print("CZPow model", query_qec_gate_cost(cz_pow))